In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=8, d_latent=64):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_latent = d_latent
        
        # Query projection stays standard
        self.q_proj = nn.Linear(d_model, d_model)
        
        # --- MLA Low-Rank Compression Projections ---
        # 1. Down-projection: Squeezes input tokens down into a tiny latent space
        self.kv_down_proj = nn.Linear(d_model, d_latent)
        
        # 2. Up-projections: Reconstruct full multi-head matrices out of the latent space
        self.k_up_proj = nn.Linear(d_latent, n_heads * self.head_dim)
        self.v_up_proj = nn.Linear(d_latent, n_heads * self.head_dim)
        
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, past_latent_cache=None):
        B, S, D = x.shape
        
        # 1. Project Queries normally
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        # q shape: (B, n_heads, S, head_dim)
        
        # 2. Compress the current input tokens into the tiny latent space
        # Instead of projecting directly to large K and V matrices, we compress first!
        latent_kv = self.kv_down_proj(x) # Shape: (B, S, d_latent)
        
        # 3. Handle the Latent KV Cache accumulation along sequence dimension (dim=1)
        if past_latent_cache is not None:
            # We concatenate the tiny latent vectors, NOT the massive full heads!
            latent_kv = torch.cat([past_latent_cache, latent_kv], dim=1)
            
        # This small latent tensor is what gets stored for the next step
        current_latent_cache = latent_kv
        
        # 4. Up-project the full accumulated latent cache back into multi-head K and V
        # Total sequence length is determined by the accumulated latent cache size
        total_seq_len = latent_kv.shape[1]
        
        k = self.k_up_proj(latent_kv).view(B, total_seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_up_proj(latent_kv).view(B, total_seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        # k & v shapes: (B, n_heads, total_seq_len, head_dim)
        
        # 5. Standard Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        
        out = torch.matmul(attn, v) # Shape: (B, n_heads, S, head_dim)
        
        # 6. Recombine heads and project out
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out), current_latent_cache

In [ ]:
i